## WP015 — Re-testing "the model adds nothing to the market" on fresh matches

See `README.md`. WP012's two model-dependent tests, repeated on the 998 Pinnacle-covered matches WP014 scored that no earlier work product ever used. **Primary (pre-declared), model = `baseline`:** P1 (log loss, market+model minus market-only, hit needs CI < 0 and both halves negative) and P2 (CLV at Pinnacle's opening odds, `tau = 0.02`, hit needs CI > 0 and both halves positive). Everything else is exploratory. No sampling.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk
from football_model.evaluation.windows import tested_rounds

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP013 = REPO / 'work_products' / 'wp013_lineup_continuity'
WP014 = REPO / 'work_products' / 'wp014_continuity_confirmation'
N_BOOT, TAU = 5000, 0.02
TAUS = [-np.inf, 0.0, 0.01, 0.02, 0.03, 0.05]

def show(df, digits=4):
    print(df.round(digits).to_string(index=False))

with open(WP014 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
old_windows = pickle.load(open(WP013 / 'cv_shared_data.pkl', 'rb'))['windows']
assert tested_rounds(windows).isdisjoint(tested_rounds(old_windows)), 'these matches overlap WP001-013 held-out rounds'
odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)
need = [f'PS{o}' for o in mk.OUTCOMES] + [f'PSC{o}' for o in mk.OUTCOMES]

def load_arm(arm):
    ckpt = pickle.load(open(WP014 / f'cv_checkpoint_{arm}.pkl', 'rb'))
    assert len(ckpt['results']) == len(windows), f'{arm}: {len(ckpt["results"])}/{len(windows)} windows'
    fx = mk.model_fixtures(df_cv, windows, ckpt)
    J = mk.join_odds(fx, odds, required_cols=need)      # raises if any joined score disagrees with the odds file
    return fx, J

fx_base, J_base = load_arm('baseline')
print(f'{len(fx_base)} new held-out matches, {len(J_base)} with Pinnacle pre-closing and closing odds; scores agree with the odds file; rounds disjoint from WP001-013')

1086 new held-out matches, 998 with Pinnacle pre-closing and closing odds; scores agree with the odds file; rounds disjoint from WP001-013


## The tests

`run_tests(arm)` computes everything for one arm's predictions: blend sweep, P1, opening-vs-closing, CLV by threshold, and the two primary verdicts.

In [2]:
def verdict(lo, hi, h1, h2, favourable):
    ok = (hi < 0 and h1 < 0 and h2 < 0) if favourable == 'negative' else (lo > 0 and h1 > 0 and h2 > 0)
    return 'HIT' if ok else 'no hit'

def run_tests(arm, J):
    print(f'=================== {arm} ===================')
    p_mod = J[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy()
    p_close = mk.devig(mk.odds_matrix(J, 'PSC'))
    p_open = mk.devig(mk.odds_matrix(J, 'PS'))
    open_odds = mk.odds_matrix(J, 'PS')
    y = mk.outcome_onehot(J['FTR'])
    y_idx = y.argmax(axis=1)
    first, second = mk.half_masks(J['date'])
    rps_pin = mk.rps(p_close, y)

    print(f'n = {len(J)}; RPS: Pinnacle close {rps_pin.mean():.4f}, Pinnacle open {mk.rps(p_open, y).mean():.4f}, model {mk.rps(p_mod, y).mean():.4f}')
    print('\nblend (1-w)*Pinnacle_close + w*model, paired RPS vs pure Pinnacle:')
    rows = []
    for w in [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0]:
        d = mk.rps(mk.blend(p_close, p_mod, w), y) - rps_pin
        m, lo, hi = mk.bootstrap_ci(d, N_BOOT)
        rows.append({'w_model': w, 'diff_vs_pinnacle': m, 'lo': lo, 'hi': hi})
    show(pd.DataFrame(rows), 5)

    # P1
    X_mkt = mk.logodds_features(p_close)
    X_both = np.hstack([X_mkt, mk.logodds_features(p_mod)])
    groups = J['window'].to_numpy()
    d1 = mk.leave_group_out_logloss(X_both, y_idx, groups) - mk.leave_group_out_logloss(X_mkt, y_idx, groups)
    m1, lo1, hi1 = mk.bootstrap_ci(d1, N_BOOT)
    h11, h12 = d1[first].mean(), d1[second].mean()

    # opening vs closing (exploratory)
    print('\nopening vs closing:')
    rows = []
    for lab, d in [('Pinnacle open - close', mk.rps(p_open, y) - rps_pin),
                   ('model - Pinnacle open', mk.rps(p_mod, y) - mk.rps(p_open, y)),
                   ('model - Pinnacle close', mk.rps(p_mod, y) - rps_pin)]:
        m, lo, hi = mk.bootstrap_ci(d, N_BOOT); rows.append({'paired RPS diff': lab, 'mean': m, 'lo': lo, 'hi': hi})
    show(pd.DataFrame(rows), 5)
    x, yy = p_mod - p_open, p_close - p_open
    sl = mk.ratio_bootstrap((x * yy).sum(axis=1), (x * x).sum(axis=1), N_BOOT)
    print(f'line-movement slope (close-open on model-open): {sl[0]:+.3f} [{sl[1]:+.3f}, {sl[2]:+.3f}]')

    # P2
    print('\nCLV of model-selected bets at Pinnacle opening odds:')
    rows = []
    for tau in TAUS:
        rows.append({'tau': tau, **mk.clv_table(open_odds, p_close, mk.edge(p_mod, open_odds) > tau, y, n_boot=N_BOOT)})
    show(pd.DataFrame(rows), 4)
    mask = mk.edge(p_mod, open_odds) > TAU
    full = mk.clv_table(open_odds, p_close, mask, n_boot=N_BOOT)
    halves = [mk.clv_table(open_odds[h], p_close[h], mask[h], n_boot=N_BOOT)['clv'] for h in (first, second)]

    summary = pd.DataFrame([
        {'test': 'P1 log loss, market+model - market-only', 'n': len(d1), 'estimate': m1, 'lo': lo1, 'hi': hi1,
         'half_1': h11, 'half_2': h12, 'result': verdict(lo1, hi1, h11, h12, 'negative')},
        {'test': f'P2 CLV per bet, tau={TAU}', 'n': full['n_bets'], 'estimate': full['clv'], 'lo': full['clv_lo'], 'hi': full['clv_hi'],
         'half_1': halves[0], 'half_2': halves[1], 'result': verdict(full['clv_lo'], full['clv_hi'], halves[0], halves[1], 'positive')},
    ])
    print(f'\n----- primary verdicts ({arm}) -----')
    show(summary, 5)
    return summary

summary_base = run_tests('baseline (PRIMARY)', J_base)

=================== baseline (PRIMARY) ===================
n = 998; RPS: Pinnacle close 0.1982, Pinnacle open 0.1993, model 0.2056

blend (1-w)*Pinnacle_close + w*model, paired RPS vs pure Pinnacle:
 w_model  diff_vs_pinnacle       lo      hi
    0.00           0.00000  0.00000 0.00000
    0.05           0.00008 -0.00011 0.00027
    0.10           0.00019 -0.00019 0.00057
    0.20           0.00051 -0.00026 0.00126
    0.30           0.00094 -0.00021 0.00207
    0.50           0.00218  0.00025 0.00406
    1.00           0.00740  0.00346 0.01129



opening vs closing:
       paired RPS diff    mean       lo      hi
 Pinnacle open - close 0.00112 -0.00015 0.00243
 model - Pinnacle open 0.00628  0.00268 0.00978
model - Pinnacle close 0.00740  0.00346 0.01129
line-movement slope (close-open on model-open): -0.028 [-0.049, -0.005]

CLV of model-selected bets at Pinnacle opening odds:


 tau  n_bets     clv  clv_lo  clv_hi     roi  roi_lo  roi_hi
-inf    2994 -0.0298 -0.0319 -0.0275 -0.0393 -0.0730 -0.0063
0.00    1220 -0.0324 -0.0385 -0.0260 -0.0203 -0.1229  0.0829
0.01    1165 -0.0324 -0.0387 -0.0257 -0.0286 -0.1337  0.0771
0.02    1115 -0.0323 -0.0389 -0.0253 -0.0350 -0.1455  0.0751
0.03    1077 -0.0328 -0.0395 -0.0258 -0.0443 -0.1556  0.0669
0.05     975 -0.0343 -0.0414 -0.0268 -0.0355 -0.1588  0.0861

----- primary verdicts (baseline (PRIMARY)) -----
                                   test    n  estimate       lo       hi   half_1   half_2 result
P1 log loss, market+model - market-only  998   0.00171 -0.00297  0.00636  0.00066  0.00275 no hit
               P2 CLV per bet, tau=0.02 1115  -0.03229 -0.03889 -0.02533 -0.02355 -0.04140 no hit


### Reference: what WP012 found on the original 361 matches

Copied from `work_products/wp012_market_inefficiency/README.md`, for comparison only.

In [3]:
show(pd.DataFrame([
    {'test': 'P1 (WP012, n=361)', 'estimate': 0.0070, 'lo': -0.0040, 'hi': 0.0183, 'result': 'no hit'},
    {'test': 'P2 (WP012, 416 bets)', 'estimate': -0.0300, 'lo': -0.0409, 'hi': -0.0191, 'result': 'no hit'},
]), 4)

                test  estimate      lo      hi result
   P1 (WP012, n=361)     0.007 -0.0040  0.0183 no hit
P2 (WP012, 416 bets)    -0.030 -0.0409 -0.0191 no hit


### Exploratory: the best arm (`continuity_lineup_loose_combo`)

Same tests, best arm, in case a slightly better model changes the picture. Not part of the pre-declared tests.

In [4]:
_, J_best = load_arm('continuity_lineup_loose_combo')
assert J_best[['date', 'home_fd', 'away_fd']].equals(J_base[['date', 'home_fd', 'away_fd']])
summary_best = run_tests('continuity_lineup_loose_combo (exploratory)', J_best)

=================== continuity_lineup_loose_combo (exploratory) ===================
n = 998; RPS: Pinnacle close 0.1982, Pinnacle open 0.1993, model 0.2044

blend (1-w)*Pinnacle_close + w*model, paired RPS vs pure Pinnacle:


 w_model  diff_vs_pinnacle       lo      hi
    0.00           0.00000  0.00000 0.00000
    0.05           0.00006 -0.00013 0.00024
    0.10           0.00014 -0.00023 0.00050
    0.20           0.00038 -0.00035 0.00111
    0.30           0.00074 -0.00036 0.00184
    0.50           0.00177 -0.00007 0.00362
    1.00           0.00626  0.00251 0.01000



opening vs closing:
       paired RPS diff    mean       lo      hi
 Pinnacle open - close 0.00112 -0.00015 0.00243
 model - Pinnacle open 0.00514  0.00171 0.00852
model - Pinnacle close 0.00626  0.00251 0.01000


line-movement slope (close-open on model-open): -0.011 [-0.034, +0.011]

CLV of model-selected bets at Pinnacle opening odds:


 tau  n_bets     clv  clv_lo  clv_hi     roi  roi_lo  roi_hi
-inf    2994 -0.0298 -0.0319 -0.0275 -0.0393 -0.0730 -0.0063
0.00    1170 -0.0304 -0.0366 -0.0239 -0.0150 -0.1208  0.0892
0.01    1117 -0.0300 -0.0366 -0.0233 -0.0161 -0.1282  0.0933
0.02    1061 -0.0302 -0.0369 -0.0232 -0.0038 -0.1203  0.1105
0.03    1025 -0.0309 -0.0378 -0.0236  0.0036 -0.1150  0.1205
0.05     951 -0.0313 -0.0385 -0.0237 -0.0209 -0.1427  0.1023

----- primary verdicts (continuity_lineup_loose_combo (exploratory)) -----
                                   test    n  estimate       lo       hi   half_1   half_2 result
P1 log loss, market+model - market-only  998   0.00067 -0.00476  0.00598 -0.00019  0.00151 no hit
               P2 CLV per bet, tau=0.02 1061  -0.03017 -0.03694 -0.02319 -0.02341 -0.03701 no hit
